In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# SQL Server connection setup
server = 'myserver'
database = 'RetailRocketDB'
conn_str = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(conn_str, fast_executemany=True)

# File path to the large dataset
file_path = r"C:\data\item_properties_part1.csv"

# Number of rows to read per chunk (optimizing memory usage)
chunksize = 50000  

# Batch size for SQL insert (must comply with SQL Server 2100 parameter limit)
insert_batch_size = 500  

# Extract column names from the first row
col_names = pd.read_csv(file_path, sep=",", nrows=0).columns.tolist()

# Read data in chunks and upload to SQL database iteratively
for chunk in pd.read_csv(file_path, sep=",", names=col_names, header=0, chunksize=chunksize):
    chunk.to_sql(
        'item_properties_part1',
        con=engine,
        if_exists='append',
        index=False,
        method='multi',
        chunksize=insert_batch_size  # Critical parameter to prevent SQL overload
    )
    print(f"{len(chunk)} rows successfully uploaded...")

print("All data successfully uploaded to SQL Server ✅")
# Data successfully ingested in batches with correct headers
